# Task 7.12 - Guard / Feature ablation on data v2

Run all four configurations in a new experiment. Do not silently reuse unverified A4 weights from Task 7.11.
A1 includes CE, KD and logit consistency; explicit v2 mode keeps CE identical across all four runs.
This design does not isolate the contribution of new data. This notebook contains no measured results yet.

Publish the fixed source before running. Prepare the audited train tar/checksum, original CFE tar,
and phase7_measured_artifacts.zip from Task 7.6 containing calibration and the original NPZ partitions.
Never create a new split to improve results. Require >=14 GiB free VRAM (recommend selecting L4 GPU with 22.5 GiB in Colab);
this threshold does not replace a real preflight and benchmark.


In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, sys, json, shutil, hashlib, tarfile, zipfile

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/Fatformer')
REPO = Path('/content/fatformer-xla')
RUNTIME = Path('/content/ablation_v2_fixed_runtime')
STAGE = RUNTIME / 'data'
EXPERIMENT = 'ablation_v2_fixed_run1'  # New name if changing any source/input/hyperparameter.
BACKUP = DRIVE / 'checkpoint' / 'phase7' / EXPERIMENT
SOURCE_REF = None  # Optional published commit SHA; resume always uses saved SHA.
CFE_ARCHIVE = DRIVE / 'datasets/test_communityforensics.tar'
MEASURED_ZIP = DRIVE / 'phase7_measured_artifacts.zip'
RUNTIME.mkdir(parents=True, exist_ok=True)
BACKUP.mkdir(parents=True, exist_ok=True)

def run(command, cwd=None):
    subprocess.run([str(x) for x in command], cwd=cwd or REPO, check=True)

def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()


## 1. Pin source for the entire experiment
Resume uses the saved commit. Preserve local source edits instead of resetting them.


In [ ]:
saved = BACKUP / 'source_commit.txt'
if saved.exists() and SOURCE_REF and saved.read_text().strip() != SOURCE_REF:
    raise ValueError('SOURCE_REF differs from saved experiment; choose a new EXPERIMENT.')
if not (REPO / '.git').exists():
    assert not REPO.exists(), 'REPO exists without Git; select another location.'
    run(['git', 'clone', '-b', 'hoang', 'https://github.com/LeeVHoangtk3/fatformer-xla.git', REPO], cwd='/content')
else:
    run(['git', 'checkout', 'HEAD', '--', 'notebooks'], cwd=REPO)
    dirty = subprocess.check_output(['git', 'status', '--porcelain', '--untracked-files=no', '--', 'src', 'configs', 'tools'], cwd=REPO, text=True)
    assert not dirty.strip(), 'Source tree has edits; preserve them before checkout.'
run(['git', 'fetch', 'origin', 'hoang'])
ref = saved.read_text().strip() if saved.exists() else (SOURCE_REF or 'origin/hoang')
run(['git', 'checkout', '--detach', ref])
SOURCE_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
assert (REPO / 'tools/run_ablation_v2.py').is_file(), 'Publish the fixed implementation before running.'
saved.write_text(SOURCE_COMMIT + '\n', encoding='ascii')
print('Pinned commit:', SOURCE_COMMIT)
run([sys.executable, '-m', 'pip', 'install', '-q', 'ftfy', 'regex', 'tqdm', 'pyyaml', 'timm', 'einops', 'pytest', 'numpy', 'Pillow', 'scikit-learn', 'pandas', 'matplotlib'])
run([sys.executable, 'tools/audit_ablation_configs.py'])


## 2. Stage original data and partitions
Keep the audited train manifest/config unchanged. Extract only the three required calibration/split artifacts.
The calibration NPZ must match the SHA256 in the measured calibration JSON. Missing evidence stops execution.


In [ ]:
STAGE.mkdir(parents=True, exist_ok=True)
archive = DRIVE / 'phase7/train_crossdomain_v2.tar'
sidecar = Path(str(archive) + '.sha256')
assert archive.is_file() and sidecar.is_file(), 'Missing audited train archive/checksum.'
expected = sidecar.read_text().split()[0]
local = RUNTIME / archive.name
if not local.exists() or digest(local) != expected:
    shutil.copy2(archive, local)
assert digest(local) == expected
# Extraction is repeatable; preflight checks every staged image after this.
with tarfile.open(local) as bundle:
    bundle.extractall(STAGE, filter='data')

assets = RUNTIME / 'pretrained'
assets.mkdir(exist_ok=True)
for name in ('ViT-L-14.pt', 'fatformer_4class_ckpt.pth'):
    source, target = DRIVE / 'pretrained' / name, assets / name
    assert source.is_file(), f'Missing {source}'
    if not target.exists() or digest(target) != digest(source):
        shutil.copy2(source, target)
    assert digest(target) == digest(source)

# The source CFE archive comes from the original acceptance dataset.
assert CFE_ARCHIVE.is_file(), f'Missing CFE archive: {CFE_ARCHIVE}'
local_cfe = RUNTIME / CFE_ARCHIVE.name
if not local_cfe.exists() or digest(local_cfe) != digest(CFE_ARCHIVE):
    shutil.copy2(CFE_ARCHIVE, local_cfe)
assert digest(local_cfe) == digest(CFE_ARCHIVE)
cfe_root = RUNTIME / 'cfe'
cfe_root.mkdir(exist_ok=True)
with tarfile.open(local_cfe) as bundle:
    bundle.extractall(cfe_root, filter='data')
# Accept exactly one extracted CFE root, rather than silently scanning an empty directory.
roots = [d.parent for d in cfe_root.rglob('1_commercial') if (d / '0_real').is_dir()]
assert len(roots) == 1, f'Need one CFE dataset root, got {roots}'
CFE_DIR = roots[0]

assert MEASURED_ZIP.is_file(), f'Missing original Task 7.6 artifact ZIP: {MEASURED_ZIP}'
partition_dir = RUNTIME / 'original_partition'
partition_dir.mkdir(exist_ok=True)
required = ['calibration/baseline.json', 'logits_baseline_clean_calib_split.npz', 'logits_baseline_clean_eval_split.npz']
with zipfile.ZipFile(MEASURED_ZIP) as bundle:
    for suffix in required:
        matches = [name for name in bundle.namelist() if name.endswith('DATASET/logs/phase7/' + suffix)]
        assert len(matches) == 1, f'Missing/ambiguous original artifact: {suffix}'
        (partition_dir / Path(suffix).name).write_bytes(bundle.read(matches[0]))
CALIBRATION = partition_dir / 'baseline.json'
CALIB_SPLIT = partition_dir / 'logits_baseline_clean_calib_split.npz'
EVAL_SPLIT = partition_dir / 'logits_baseline_clean_eval_split.npz'

# Bind immutable staged inputs to the experiment across Colab sessions.
inputs = {'train_archive_sha256': expected, 'cfe_archive_sha256': digest(local_cfe),
          'calibration_sha256': digest(CALIBRATION), 'calibration_split_sha256': digest(CALIB_SPLIT),
          'evaluation_split_sha256': digest(EVAL_SPLIT)}
marker = BACKUP / 'staged_inputs.json'
if marker.exists():
    assert json.loads(marker.read_text()) == inputs, 'Inputs changed; choose a new EXPERIMENT.'
marker.write_text(json.dumps(inputs, indent=2) + '\n')
print('Staged inputs; full preflight still required.')


## 3. Train/resume, verify checkpoints and evaluate held-out images
Each configuration needs preflight and a benchmark on the current GPU. Resume requires a full latest snapshot
with checksum and matching context. A best file and CSV do not prove that the epoch horizon was completed.
Non-finite gradients stop the run; investigate AMP in a separate experiment without changing other arms silently.
Worst-group deltas remain descriptive without independent uncertainty estimates. Paired CFE bootstrap intervals
measure image uncertainty conditional on training seed 42, not variation between training seeds.


In [ ]:
run(['git', 'checkout', 'HEAD', '--', 'notebooks'], cwd=REPO)
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
run([sys.executable, 'tools/run_ablation_v2.py',
     '--data_root', STAGE, '--assets', assets, '--cfe_dir', CFE_DIR,
     '--calibration', CALIBRATION, '--calibration_split', CALIB_SPLIT,
     '--evaluation_split', EVAL_SPLIT, '--run_root', RUNTIME / EXPERIMENT,
     '--backup_root', BACKUP])
print('Artifacts saved. Review run verification and limitations; G7.2 is not automatically passed.')


In [ ]:
import pandas as pd
from IPython.display import display, Image
summary_dir = RUNTIME / EXPERIMENT / 'summary'
display(pd.read_csv(summary_dir / 'ablation_mini_v2.csv'))
report = json.loads((summary_dir / 'ablation_mini_v2.json').read_text(encoding='utf-8'))
print(report['summary'])
print(json.dumps(report['worst_group_descriptive_contrasts'], ensure_ascii=False, indent=2))
display(Image(filename=str(summary_dir / 'm7_ablation.png')))
